# Collect AVH Data for Analysis
The following code brings together data generated and collected by Feng, Changye, and George into a single dataframe for future analysis. Right now the focus is on transcript level information for the sandwich analysis.

TODO

    1) How to handle NAs? Can implement smarter NA handling by only removing rows that have NAs in the columns we care about
    2) Log of WER
    3) Add in pause proportion
    4) Do not use SNR for WER

## Import libraries and functions

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
# import statsmodels.api as sm
# from statsmodels.regression.mixed_linear_model import MixedLM
import numpy as np
from io import StringIO
from pathlib import Path

# directory_path = Path("another_new_directory")

# directory_path.mkdir(parents=True, exist_ok=True)
# print(f"Directory '{directory_path}' ensured to exist.")
out_folder = "/edata/obdw/sandwich_analysis_data/"

In [2]:
def process_uid(series: pd.SparseDtype) -> pd.Series:
    """
    Extract unique pid from file names in a pandas Series. From Changye Li

    Args:
        series (pd.Series): Series containing file names of the transcripts

    Returns:
        pd.Series: Series with processed PIDs as integers
    """
    # Split the strings and get the first element (pid part)
    pids = series.str.split('-').str[0]
    
    # Clean up the pid strings
    pids = (pids.str.rstrip('@avh')
                .str.lstrip('u')
                .str.lstrip('0')
                .astype(int))
    
    return pids

## Read in Data

In [3]:
ptcpt_baseline_path = "/edata/changye/avh-data/avh_record_baseline_kwargs.jsonl"# I wil use this file as the main
rcd_whisperx_scores_path = "/edata/feng/avh/whisperx_merged_with_manual.json" #"/edata/feng/avh/whisperx_full_cleaned.json"
rcd_snr_path = "/edata/changye/avh-new/data/snr.csv"
rcd_mos_path = "/edata/changye/avh-data/pred_mos.csv"
rcd_coh_recording_level_path = "/edata/george/debias_merged_new.csv"




In [4]:
baseline_df = pd.read_json(ptcpt_baseline_path, lines=True)
baseline_df = baseline_df.set_index("pid", drop=True).rename_axis("file")
baseline_df = baseline_df.rename(columns={"uid":"pid"})
baseline_df.shape

(2904, 128)

In [5]:
baseline_df = pd.read_json(ptcpt_baseline_path, lines=True)
baseline_df = baseline_df.set_index("pid", drop=True).rename_axis("file")
baseline_df = baseline_df.rename(columns={"uid":"pid"})

rcd_whisperx_scores_df = pd.read_json(rcd_whisperx_scores_path, lines=True)
rcd_whisperx_scores_df = rcd_whisperx_scores_df.set_index("file", drop=True)#.rename_axis("file")
# rcd_whisperx_scores_df["pid"] = process_uid(rcd_whisperx_scores_df.pid)


rcd_snr_df = pd.read_csv(rcd_snr_path)
rcd_snr_df = rcd_snr_df.set_index("file")
rcd_snr_df["pid"] = process_uid(rcd_snr_df.index)

rcd_mos_df = pd.read_csv(rcd_mos_path)
rcd_mos_df = pd.DataFrame(rcd_mos_df.groupby(['file_name'])['pred_mos'].mean())
rcd_mos_df["pid"] = process_uid(rcd_mos_df.index)


rcd_level_coh_df = pd.read_csv(rcd_coh_recording_level_path)# Similar to above but for recordings.
rcd_level_coh_df["file"] = rcd_level_coh_df.file.apply(lambda x: x.strip(".txt"))
rcd_level_coh_df = rcd_level_coh_df.set_index("file")
rcd_level_coh_df["pid"] = process_uid(rcd_level_coh_df.index)


In [6]:
list(rcd_whisperx_scores_df.columns)

['segments',
 'prediction',
 'time_diffs',
 'pause_proportion',
 'pid',
 'record',
 'text',
 'audio',
 'wer',
 'cer']

In [7]:
list(rcd_snr_df.columns)

['snr', 'pid']

In [8]:
list(rcd_mos_df.columns)

['pred_mos', 'pid']

In [9]:
list(rcd_level_coh_df.columns)

['text',
 'wordCoherenceSeq',
 'wordCoherenceStaticCentroid',
 'wordCoherenceCumulativeCentroid',
 'phraseCoherenceSeq',
 'phraseCoherenceStaticCentroid',
 'phraseCoherenceCumulativeCentroid',
 'sentCoherenceSeq',
 'sentCoherenceStaticCentroid',
 'sentCoherenceCumulativeCentroid',
 'sentCoherenceWeightedSeq',
 'sentCoherenceWeightedStaticCentroid',
 'sentCoherenceWeightedCumulativeCentroid',
 'wordCoherenceBertSumSeq',
 'wordCoherenceBertSumStaticCentroid',
 'wordCoherenceBertSumCumulativeCentroid',
 'phraseCoherenceBertSumSeq',
 'phraseCoherenceBertSumStaticCentroid',
 'phraseCoherenceBertSumCumulativeCentroid',
 'sentCoherenceBertSumSeq',
 'sentCoherenceBertSumStaticCentroid',
 'sentCoherenceBertSumCumulativeCentroid',
 'sentCoherenceBert2ndLayerSeq',
 'sentCoherenceBert2ndLayerStaticCentroid',
 'sentCoherenceBert2ndLayerCumulativeCentroid',
 'sentCoherenceBertClsSeq',
 'sentCoherenceBertClsStaticCentroid',
 'sentCoherenceBertClsCumulativeCentroid',
 'sentCoherenceSentBertSeq',
 'sen

In [10]:
list(rcd_snr_df.columns)

['snr', 'pid']

In [11]:
print("Baseline scores shape", baseline_df.shape)
print("Whisper scores shape", rcd_whisperx_scores_df.shape)
print("SNR shape", rcd_snr_df.shape)
print("MOS shape", rcd_mos_df.shape)
print("COH shape", rcd_level_coh_df.shape)

Baseline scores shape (2904, 128)
Whisper scores shape (2994, 10)
SNR shape (3003, 2)
MOS shape (2981, 2)
COH shape (3003, 133)


## Clean Data

### Check transcript file names

In [12]:
baseline_files = set(baseline_df.index)
whisper_files = set(rcd_whisperx_scores_df.index)
snr_files = set(rcd_snr_df.index)
mos_files = set(rcd_mos_df.index)
coh_files = set(rcd_level_coh_df.index)

file_names_union = whisper_files.union(snr_files).union(mos_files).union(coh_files).union(baseline_files)
file_names_intersection = whisper_files.intersection(snr_files).intersection(mos_files).intersection(coh_files).intersection(baseline_files)
print("There are a total of {:,} file names and only {:,} at the intersection".format(len(file_names_union), len(file_names_intersection)))

There are a total of 3,007 file names and only 2,886 at the intersection


### Remove duplicated columns
Not all will be removed but this should clean it up some

In [13]:
def remove_duplicated_columns(main_df, secondary_df, extra_cols_to_drop, protected_cols = set(["pid"])):
    """
    Description: removes duiplicated columns from secondary_df which main_df already contains
    Input:
        main_df (pandas df): dataframe
        secondary_df (pandas df): dataframe
        extra_cols_to_drop (list or set): Additional column names to drop from extra cols. Useful if you know cols
            are duplicates but with different names
        protected_cols (list or set): Columns not to drop.  THIS IS NO LONGER USED AND DOES NOT WORK ANYWAY
    Output:
        updated_df: (pandas df): same as secondary_df, but with duplicated columns removed
    TODO
        1)
    """

    intersection_columns = set(main_df).intersection(secondary_df).union(extra_cols_to_drop)
    print(intersection_columns)
    print("cols dropped", len(intersection_columns))
    updated_df = secondary_df.drop(intersection_columns, axis=1)
    print("new shape", updated_df.shape)
    return(updated_df)

In [14]:
coh_cols_to_drop = ['contact',
 'temper-outbursts',
 'feeling-blocked',
 'worrying-too-much',
 'easily-hurt',
 'feeling-watched',
 'feeling-tense',
 'heavy-feelings-in-arms-legs',
 'feeling-nervous',
 'feeling-lonely',
 'frequency',
 'bad-voices',
 'volume-of-voices',
 'voices-length',
 'interference-in-activities',
 'distressing-voices',
 'worthless-useless-voices',
 'clarity-of-voices',
 'follow-voices-orders',
 'time-of-day-of-voices',
 'social-situations',
 'where-are-the-voices',
 'typical-week',
 'if-no-explanation',
 'work-school-disruption',
 'social-leisure-disruption',
 'home-family-disruption',
 'school-work-missed',
 'less-productive-days',
#  'not-worked-for-other-reasons',# not removed since this is one George suggested to keep
 'little-interest-or-pleasure',
 'feeling-depressed',
 'trouble-sleeping',
 'feeling-tired',
 'appetite',
 'feeling-bad-about-self',
 'trouble-concentrating',
 'slow-fast-speaking',
 'suicidal-thoughts',
 'impact-on-your-life', 'text']

In [15]:
print("Removing columns from coh df")
rcd_level_coh_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_level_coh_df, extra_cols_to_drop=coh_cols_to_drop)
print("\n\nRemoving columns from whisperx df")
rcd_whisperx_scores_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_whisperx_scores_df, extra_cols_to_drop=["text"])
print("\n\nRemoving columns from mos df")
rcd_mos_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_mos_df, extra_cols_to_drop=[])
print("\n\nRemoving columns from snr df")
rcd_snr_df = remove_duplicated_columns(main_df=baseline_df, secondary_df=rcd_snr_df, extra_cols_to_drop=[])

Removing columns from coh df
{'follow-voices-orders', 'phone', 'heavy-feelings-in-arms-legs', 'availability', 'education', 'impact-on-your-life', 'sexuality', 'home-family-disruption', 'hispanic', 'wearable-tech', 'little-interest-or-pleasure', 'feeling-lonely', 'social-situations', 'feeling-watched', 'appetite', 'marital-status', 'distressing-voices', 'location', 'bad-voices', 'if-no-explanation', 'computer', 'suicidal-thoughts', 'social-leisure-disruption', 'frequency', 'verify-hearing-voices', 'trouble-concentrating', 'ah_freq', 'comp_q5', 'voices-length', 'gender', 'easily-hurt', 'comp_q3', 'time-of-day-of-voices', 'email', 'who-knows', 'social-media', 'typical-week', 'feeling-blocked', 'clarity-of-voices', 'race', 'tablet', 'type-of-treatments', 'data_plan', 'feeling-tired', 'interference-in-activities', 'trouble-sleeping', 'employment-status', 'where-are-the-voices', 'how-often', 'how-long', 'comp_q4', 'living-status', 'basic-cell', 'feeling-depressed', 'school-work-missed', 'vol

## Merge Data

In [16]:
main_analysis_df = pd.concat([baseline_df, rcd_whisperx_scores_df, rcd_snr_df, rcd_mos_df, rcd_level_coh_df], axis=1, join="inner")
assert main_analysis_df.shape[0] == len(file_names_intersection)
print(main_analysis_df.shape)

(2886, 189)


## Remove NA
I am ignoring certain columns I do not expect to use. Like many of the coherence columns. Otherwise we would be dropping many columns with NAs that we would never use.

In [17]:
columns2ignore_na = ["not-worked-for-other-reasons"]
for column_name in main_analysis_df.columns:
    if "Coherence" in column_name and column_name != "sentCoherenceSentBertCumulativeCentroid":# the one we care about is sentCoherenceSentBertCumulativeCentroid
        columns2ignore_na.append(column_name)
# Code below just to look at columns with na
#  for x, y in zip(main_analysis_df.columns, list(main_analysis_df.isna().sum())):
#     if x in columns2ignore_na:
#         continue
#     print(x, y)
main_analysis_df.drop(columns2ignore_na, axis=1, inplace=True)
main_analysis_df.dropna(axis=0, inplace=True)
main_analysis_df


,pid,record,transcription,audio,prediction,wer,cer,record_date,in_person,days of data,...,LSC,density,degree_average,degree_std,L1,id,account_id,account_status,page_progress,Tracking
u00001870@avh-20190708-17,1870,17,"In addition, I've had tons of problems with my...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,"In addition, I've had tons of problems with my...",7.529412,3.307304,2019-07-08T23:04:00.000,1.0,31.0,...,206,0.010040,4.116505,4.408291,0,1870,1870,study_finished,downloading,FINISHED
u00001870@avh-20190707-11,1870,11,"And so initially, this would have been 2017. U...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,"And, so, initially, when this would have been ...",7.915567,4.319762,2019-07-07T12:53:00.000,1.0,31.0,...,185,0.010985,4.064516,4.837814,0,1870,1870,study_finished,downloading,FINISHED
u00001553@avh-20190207-3,1553,3,Still hearing the whispers. Some forceful talk...,/edata/avh_data/Audio_dump/u00001553@avh-20190...,Still hearing the whispers. Some forceful talk...,26.666667,9.973753,2019-02-07T18:58:00.000,1.0,31.0,...,52,0.020360,2.483871,1.456265,1,1553,1553,study_finished,downloading,FINISHED
u00001788@avh-20190415-1,1788,1,I'm here. The voices try and tell me some thin...,/edata/avh_data/Audio_dump/u00001788@avh-20190...,"Hi, I'm here. The voice is trying to tell me s...",41.379310,27.118644,2019-04-15T06:06:00.000,2.0,25.0,...,19,0.060606,2.545455,1.339175,0,1788,1788,app_installed,downloading,FINISHED
u00002319@avh-20190901-1,2319,1,The voices today are the same as yesterday tel...,/edata/avh_data/Audio_dump/u00002319@avh-20190...,"The voices today are the same as yesterday, te...",0.000000,0.000000,2019-09-01T07:55:00.000,1.0,31.0,...,43,0.027211,2.612245,1.562524,1,2319,2319,study_finished,downloading,FINISHED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,15,"The problem that I was having, uh, I mean, I h...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,The problem that i was having a...I have long ...,27.950311,15.113452,2019-07-08T22:58:00.000,1.0,31.0,...,215,0.010389,4.446512,5.968575,1,1870,1870,study_finished,downloading,FINISHED
u00000866@avh-20181001-3,866,3,"Um, this is my third recording of the evening....",/edata/avh_data/Audio_dump/u00000866@avh-20181...,"Um, this is my third recording of the evening,...",39.733840,28.162772,2018-10-01T19:11:00.000,1.0,31.0,...,223,0.010605,4.708520,6.451632,0,866,866,study_finished,downloading,FINISHED
u00001425@avh-20190127-1,1425,1,"Good morning. Uh, I didn't hear the voices whe...",/edata/avh_data/Audio_dump/u00001425@avh-20190...,"Good morning. Uh, I didn't hear the voices whe...",5.405405,3.499079,2019-01-27T09:26:00.000,1.0,33.0,...,60,0.024876,3.283582,2.380305,0,1425,1425,app_installed,downloading,FINISHED
u00002182@avh-20190813-3,2182,3,"I believe that I heard God say , but I heard G...",/edata/avh_data/Audio_dump/u00002182@avh-20190...,"I believe that I heard God say, in previous, I...",128.571429,126.984127,2019-08-13T18:42:00.000,1.0,31.0,...,19,0.060606,2.545455,1.269476,0,2182,2182,study_finished,downloading,FINISHED


In [18]:
main_analysis_df.to_csv(out_folder + "main_merged_sandwich_analysis_data.csv")

In [19]:
temp = pd.read_csv(out_folder + "main_merged_sandwich_analysis_data.csv", index_col=0)

In [20]:
temp

,pid,record,transcription,audio,prediction,wer,cer,record_date,in_person,days of data,...,LSC,density,degree_average,degree_std,L1,id,account_id,account_status,page_progress,Tracking
u00001870@avh-20190708-17,1870,17,"In addition, I've had tons of problems with my...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,"In addition, I've had tons of problems with my...",7.529412,3.307304,2019-07-08T23:04:00.000,1.0,31.0,...,206,0.010040,4.116505,4.408291,0,1870,1870,study_finished,downloading,FINISHED
u00001870@avh-20190707-11,1870,11,"And so initially, this would have been 2017. U...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,"And, so, initially, when this would have been ...",7.915567,4.319762,2019-07-07T12:53:00.000,1.0,31.0,...,185,0.010985,4.064516,4.837814,0,1870,1870,study_finished,downloading,FINISHED
u00001553@avh-20190207-3,1553,3,Still hearing the whispers. Some forceful talk...,/edata/avh_data/Audio_dump/u00001553@avh-20190...,Still hearing the whispers. Some forceful talk...,26.666667,9.973753,2019-02-07T18:58:00.000,1.0,31.0,...,52,0.020360,2.483871,1.456265,1,1553,1553,study_finished,downloading,FINISHED
u00001788@avh-20190415-1,1788,1,I'm here. The voices try and tell me some thin...,/edata/avh_data/Audio_dump/u00001788@avh-20190...,"Hi, I'm here. The voice is trying to tell me s...",41.379310,27.118644,2019-04-15T06:06:00.000,2.0,25.0,...,19,0.060606,2.545455,1.339175,0,1788,1788,app_installed,downloading,FINISHED
u00002319@avh-20190901-1,2319,1,The voices today are the same as yesterday tel...,/edata/avh_data/Audio_dump/u00002319@avh-20190...,"The voices today are the same as yesterday, te...",0.000000,0.000000,2019-09-01T07:55:00.000,1.0,31.0,...,43,0.027211,2.612245,1.562524,1,2319,2319,study_finished,downloading,FINISHED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,15,"The problem that I was having, uh, I mean, I h...",/edata/avh_data/Audio_dump/u00001870@avh-20190...,The problem that i was having a...I have long ...,27.950311,15.113452,2019-07-08T22:58:00.000,1.0,31.0,...,215,0.010389,4.446512,5.968575,1,1870,1870,study_finished,downloading,FINISHED
u00000866@avh-20181001-3,866,3,"Um, this is my third recording of the evening....",/edata/avh_data/Audio_dump/u00000866@avh-20181...,"Um, this is my third recording of the evening,...",39.733840,28.162772,2018-10-01T19:11:00.000,1.0,31.0,...,223,0.010605,4.708520,6.451632,0,866,866,study_finished,downloading,FINISHED
u00001425@avh-20190127-1,1425,1,"Good morning. Uh, I didn't hear the voices whe...",/edata/avh_data/Audio_dump/u00001425@avh-20190...,"Good morning. Uh, I didn't hear the voices whe...",5.405405,3.499079,2019-01-27T09:26:00.000,1.0,33.0,...,60,0.024876,3.283582,2.380305,0,1425,1425,app_installed,downloading,FINISHED
u00002182@avh-20190813-3,2182,3,"I believe that I heard God say , but I heard G...",/edata/avh_data/Audio_dump/u00002182@avh-20190...,"I believe that I heard God say, in previous, I...",128.571429,126.984127,2019-08-13T18:42:00.000,1.0,31.0,...,19,0.060606,2.545455,1.269476,0,2182,2182,study_finished,downloading,FINISHED


In [21]:
main_analysis_df.shape

(2863, 153)

## Identify Features for Analysis
The next step is to identify subsets of features to use for sandwich analysis. Listing the features and response variable out should not be hard for each analysis. Removing nan values will depend more on the response variable I believe, since nan categorical variables can be coded up in one-hot encoding. However it might be wise to remove rows with any nan values.

### Options for final cleaning

1) Remove nan values from each dataframe on a per analysis criteria
2) Remove nan values from the overall dataframe so each analysis is working with the same dataframe.
3) A hybrid approach of the above where rows are removed based off of nan values in the features from the overall dataframe and then nan values are removed in the response variable on a per analysis criteria.

## Feature Engineering
As of now, this is only one-hot encoding code I have from the previous analysis

In [22]:
list(main_analysis_df.columns)

['pid',
 'record',
 'transcription',
 'audio',
 'prediction',
 'wer',
 'cer',
 'record_date',
 'in_person',
 'days of data',
 'audio diary entries',
 'age',
 'gender',
 'phone',
 'data_plan',
 'ah_freq',
 'location',
 'availability',
 'future_contact',
 'comp_q1',
 'comp_q2',
 'comp_q3',
 'comp_q4',
 'comp_q5',
 'race',
 'hispanic',
 'marital-status',
 'sexuality',
 'education',
 'employment-status',
 'living-status',
 'basic-cell',
 'smartphone',
 'tablet',
 'computer',
 'wearable-tech',
 'email',
 'social-media',
 'verify-hearing-voices',
 'how-often',
 'how-long',
 'diagnoses',
 'dx-alzheimers-parkinsons',
 'dx-bipolar-disorder',
 'dx-depression',
 'dx-tbi',
 'dx-migraines',
 'dx-schizoaffective-disorder',
 'dx-schizophrenia',
 'dx-ptsd',
 'dx-substance-use',
 'dx-seizures',
 'dx-none-of-the-above',
 'dx-other',
 'used-treatment',
 'type-of-treatments',
 'tx-one-on-one',
 'tx-online',
 'tx-group',
 'tx-partial-hospitalization',
 'tx-alcohol-drug-rehab',
 'tx-telepsychiatry',
 'tx-re

In [23]:
Y_WER = pd.concat([main_analysis_df.wer, np.log(main_analysis_df.wer + 1e-8)],axis =1, keys = ["wer", "log_wer"]) #main_analysis_df.wer
Y_COH = main_analysis_df.sentCoherenceSentBertCumulativeCentroid # Need to check if this is still the best one to use.

In [24]:
binmappers = {
                "education":{
                    2.0: 1.0,# 1.0 means up to codes for education - data dictionary for things like HS diploma, College degree, etc.
                    3.0: 1.0,
                    4.0: 1.0,
                    5.0: 2.0,
                    6.0: 2.0,
                    7.0: 3.0,
                    8.0: 3.0,
                    np.nan:np.nan,
                }
           }
def bin_age(age):
    """
    Description: Bins all age value from an pandas series. To be used in an apply function
    Inputs:
        ages (float): Float age
    Outputs:
        age_bin (int): Int category for age bin
    TODO:
        1)
    """
    if np.isnan(age):
        age_bin = 0.0
    elif age == 999.0:
        age_bin = 0.0
    # elif age < 18.0:
    #     age_bin = 0.0
    elif age < 30.0:
        age_bin = 1.0
    elif age < 45.0:
        age_bin = 2.0
    elif age < 65.0:
        age_bin = 3.0
    else:
        age_bin = 4.0
    return(age_bin)

def replaced_age_with_binning(df):
    """
    Description: Wrapper for bin_age where the original age columns is dropped
    Inputs:
        df (pandas df): Pandas dataframe
    Outputs:
        df (pandas df): Same as input, but with age removed, and binned_age added
    TODO:
        1)
    """
    temp_df = df.copy(deep=True)
    temp_df["binned_age"] = temp_df.age.apply(lambda x: bin_age(x))
    temp_df.drop("age", axis=1, inplace=True)
    return(temp_df)
def bin_substance_use(data_df):
    """
    Decription: Bins a participants substance use. Only done for heroin and opioids
    Inputs:
         data_df (pandas DF): Dataframe containing substance use columns below
            "opioids","marijuana","alcohol","steroids","cocaine","heroin","nicotine","meth","ketamine","acid","bath-salts"
    Outputs:
    TODO
        1)
    """

    data_copy_df = data_df.copy(deep=True)
    # all_types = ["opioids","marijuana","alcohol","steroids","cocaine","heroin","nicotine","meth","ketamine","acid","bath-salts"]
    substances_of_interest = ["opioids","heroin"]
    # type_2 = ["cocaine","heroin","nicotine","meth","ketamine","acid","bath-salts"]
    data_copy_df.loc[:,substances_of_interest] = pd.DataFrame(data_copy_df.loc[:,substances_of_interest] == 1, dtype=np.float64)
    opiods_opiates_1_use = pd.Series(data_copy_df.loc[:,substances_of_interest].sum(axis=1) > 1, dtype=np.float64)
    data_copy_df.drop(labels = substances_of_interest, axis = 1, inplace = True)
    data_copy_df["opioids-opiates"] = opiods_opiates_1_use
    return(data_copy_df)
def one_hot_encode(df, cols2encode):
    """
    Description: Encodes the specified columns into one hot encoding.
    Inputs:
        df (pandas df): Dataframe with the columns to encode. Can have more.
        cls2encode (list): List of column names.
    Outputs:
        one_hot_encoded_df (pandas df): one hot encoded dataframe. 
    TODO:
        1)
    """
    enc = OneHotEncoder(drop="first")
    enc.fit(df[cols2encode])# you will have access to feature names with get_feature_names_out() 
    print(enc.categories_)
    print(enc.get_feature_names_out())
    one_hot_encoded_df =  pd.DataFrame(enc.transform(df[cols2encode]).toarray(), columns = enc.get_feature_names_out())
    one_hot_encoded_df = one_hot_encoded_df.set_index(df.index)
    return(one_hot_encoded_df)
    # X = minimum_analysis_df.drop(["race", "gender", "binned_age"], axis=1)
    # X_min_analysis = pd.concat([X, X_one_hot, Y_WER, Y_COH], axis=1, join="inner", ignore_index=False)

### Create minimum analysis data: Basic
with race, age, and gender.

In [25]:
cols2encode=["race", "gender", "binned_age"]
# X_temp = X_temp.replace(binmappers)# Mapping education to higher level categories
# X_temp.rename(columns={"education":"education_binned"},inplace=True)
minimum_analysis_df = replaced_age_with_binning(main_analysis_df)
minimum_analysis_df = minimum_analysis_df.loc[:,cols2encode + ["pid"]]
# minimum_analysis_df
cols2encode=["race", "gender", "binned_age"]
X_one_hot = one_hot_encode(df=minimum_analysis_df, cols2encode=cols2encode)
X = minimum_analysis_df.drop(cols2encode, axis=1)
X_min_analysis = pd.concat([X, X_one_hot, Y_WER, Y_COH], axis=1, join="inner", ignore_index=False)
X_min_analysis.to_csv(out_folder + "basic_analysis.csv")
X_min_analysis


[array([  1.,   2.,   4.,   5.,   6., 999.]), array([1., 2., 3., 4., 5.]), array([0., 1., 2., 3., 4.])]
['race_2.0' 'race_4.0' 'race_5.0' 'race_6.0' 'race_999.0' 'gender_2.0'
 'gender_3.0' 'gender_4.0' 'gender_5.0' 'binned_age_1.0' 'binned_age_2.0'
 'binned_age_3.0' 'binned_age_4.0']


,pid,race_2.0,race_4.0,race_5.0,race_6.0,race_999.0,gender_2.0,gender_3.0,gender_4.0,gender_5.0,binned_age_1.0,binned_age_2.0,binned_age_3.0,binned_age_4.0,wer,log_wer,sentCoherenceSentBertCumulativeCentroid
u00001870@avh-20190708-17,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.529412,2.018817,0.062340
u00001870@avh-20190707-11,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.915567,2.068831,0.283842
u00001553@avh-20190207-3,1553,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,26.666667,3.283414,0.488357
u00001788@avh-20190415-1,1788,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,41.379310,3.722781,0.698150
u00002319@avh-20190901-1,2319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.000000,-18.420681,0.783950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,27.950311,3.330428,0.241492
u00000866@avh-20181001-3,866,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,39.733840,3.682203,0.166712
u00001425@avh-20190127-1,1425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,5.405405,1.687399,0.391968
u00002182@avh-20190813-3,2182,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,128.571429,4.856485,0.700575


### Create analysis data: Basic+
Everything in Basic with MOS and SNR added

In [26]:
X_basic_plus = pd.concat([X_min_analysis, main_analysis_df.loc[:, ["snr", "pred_mos", "pause_proportion"]]], axis=1, join="inner", ignore_index=False)
assert X_basic_plus.shape[0] == X_min_analysis.shape[0], "Seem to have lost some columns during the join"
X_basic_plus.to_csv(out_folder + "basic_plus_analysis.csv")
X_basic_plus

,pid,race_2.0,race_4.0,race_5.0,race_6.0,race_999.0,gender_2.0,gender_3.0,gender_4.0,gender_5.0,binned_age_1.0,binned_age_2.0,binned_age_3.0,binned_age_4.0,wer,log_wer,sentCoherenceSentBertCumulativeCentroid,snr,pred_mos,pause_proportion
u00001870@avh-20190708-17,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.529412,2.018817,0.062340,5.375672,3.936099,0.083113
u00001870@avh-20190707-11,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.915567,2.068831,0.283842,5.239416,4.195542,0.051594
u00001553@avh-20190207-3,1553,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,26.666667,3.283414,0.488357,11.164604,4.162455,0.252369
u00001788@avh-20190415-1,1788,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,41.379310,3.722781,0.698150,9.495262,2.420873,0.424867
u00002319@avh-20190901-1,2319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.000000,-18.420681,0.783950,14.463228,3.313525,0.089726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,27.950311,3.330428,0.241492,3.002743,4.257866,0.029134
u00000866@avh-20181001-3,866,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,39.733840,3.682203,0.166712,7.682102,4.195727,0.085740
u00001425@avh-20190127-1,1425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,5.405405,1.687399,0.391968,1.235781,4.747826,0.091976
u00002182@avh-20190813-3,2182,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,128.571429,4.856485,0.700575,3.069747,1.750710,0.086136


### Create analysis data: Basic+ Clinical
Everything in Basic+ with the additions of phq9 total, hpsvq total, scl global average

In [27]:
main_analysis_df.loc[:, ["phq9-total", "hpsvq-total-score", "scl-avg-global-score"]]

,phq9-total,hpsvq-total-score,scl-avg-global-score
u00001870@avh-20190708-17,10.0,31.0,2.444
u00001870@avh-20190707-11,10.0,31.0,2.444
u00001553@avh-20190207-3,24.0,22.0,3.222
u00001788@avh-20190415-1,22.0,21.0,1.556
u00002319@avh-20190901-1,18.0,18.0,1.889
...,...,...,...
u00001870@avh-20190708-15,10.0,31.0,2.444
u00000866@avh-20181001-3,11.0,28.0,2.444
u00001425@avh-20190127-1,18.0,28.0,2.222
u00002182@avh-20190813-3,25.0,32.0,3.667


In [28]:
X_basic_plus_clin = pd.concat([X_basic_plus, main_analysis_df.loc[:, ["phq9-total", "hpsvq-total-score", "scl-avg-global-score"]]], axis=1, join="inner", ignore_index=False)
assert X_basic_plus_clin.shape[0] == X_basic_plus.shape[0], "Seem to have lost some columns during the join"
X_basic_plus_clin.to_csv(out_folder + "basic_plus_clinical_analysis.csv")
X_basic_plus_clin

,pid,race_2.0,race_4.0,race_5.0,race_6.0,race_999.0,gender_2.0,gender_3.0,gender_4.0,gender_5.0,...,binned_age_4.0,wer,log_wer,sentCoherenceSentBertCumulativeCentroid,snr,pred_mos,pause_proportion,phq9-total,hpsvq-total-score,scl-avg-global-score
u00001870@avh-20190708-17,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,7.529412,2.018817,0.062340,5.375672,3.936099,0.083113,10.0,31.0,2.444
u00001870@avh-20190707-11,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,7.915567,2.068831,0.283842,5.239416,4.195542,0.051594,10.0,31.0,2.444
u00001553@avh-20190207-3,1553,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,26.666667,3.283414,0.488357,11.164604,4.162455,0.252369,24.0,22.0,3.222
u00001788@avh-20190415-1,1788,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,41.379310,3.722781,0.698150,9.495262,2.420873,0.424867,22.0,21.0,1.556
u00002319@avh-20190901-1,2319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,-18.420681,0.783950,14.463228,3.313525,0.089726,18.0,18.0,1.889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,27.950311,3.330428,0.241492,3.002743,4.257866,0.029134,10.0,31.0,2.444
u00000866@avh-20181001-3,866,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,39.733840,3.682203,0.166712,7.682102,4.195727,0.085740,11.0,28.0,2.444
u00001425@avh-20190127-1,1425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.405405,1.687399,0.391968,1.235781,4.747826,0.091976,18.0,28.0,2.222
u00002182@avh-20190813-3,2182,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,128.571429,4.856485,0.700575,3.069747,1.750710,0.086136,25.0,32.0,3.667


### Create analysis data: Basic+ Clinical and SDH
Everything in Basic+ Clinical model with the additions of SDH features (sexuality, employment-status, education, and all drug data)

In [29]:
temp = bin_substance_use(data_df=main_analysis_df).loc[:, ["opioids-opiates", "marijuana","alcohol","steroids","cocaine","nicotine","meth","ketamine","acid","bath-salts", "sexuality", "employment-status", "education"]]
list(temp.columns)
temp = temp.replace(binmappers)# Mapping education to higher level categories
temp.rename(columns={"education":"education_binned"},inplace=True)
cols2encode=["sexuality", "employment-status", "education_binned"]
X_one_hot = one_hot_encode(df=temp, cols2encode=cols2encode)
X_basic_plus_clin_sdh = pd.concat([X_basic_plus_clin, X_one_hot], axis=1, join="inner", ignore_index=False)
assert X_basic_plus_clin_sdh.shape[0] == X_basic_plus_clin.shape[0], "Seem to have lost some columns during the join"
X_basic_plus_clin_sdh.to_csv(out_folder + "basic_plus_clinical_sdh_analysis.csv")
X_basic_plus_clin_sdh

[array([  1.,   2.,   3.,   4., 999.]), array([  1.,   2.,   3., 999.]), array([1., 2., 3.])]
['sexuality_2.0' 'sexuality_3.0' 'sexuality_4.0' 'sexuality_999.0'
 'employment-status_2.0' 'employment-status_3.0' 'employment-status_999.0'
 'education_binned_2.0' 'education_binned_3.0']


,pid,race_2.0,race_4.0,race_5.0,race_6.0,race_999.0,gender_2.0,gender_3.0,gender_4.0,gender_5.0,...,scl-avg-global-score,sexuality_2.0,sexuality_3.0,sexuality_4.0,sexuality_999.0,employment-status_2.0,employment-status_3.0,employment-status_999.0,education_binned_2.0,education_binned_3.0
u00001870@avh-20190708-17,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2.444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u00001870@avh-20190707-11,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2.444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u00001553@avh-20190207-3,1553,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,3.222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u00001788@avh-20190415-1,1788,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.556,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
u00002319@avh-20190901-1,2319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
u00001870@avh-20190708-15,1870,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2.444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u00000866@avh-20181001-3,866,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.444,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
u00001425@avh-20190127-1,1425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
u00002182@avh-20190813-3,2182,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,3.667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
# Summary Statistics for X_basic_plus_clin_sdh
print("=" * 80)
print("SUMMARY STATISTICS: X_basic_plus_clin_sdh")
print("=" * 80)

print("\n1. DATASET SHAPE AND SIZE:")
print(f"   - Rows: {X_basic_plus_clin_sdh.shape[0]}")
print(f"   - Columns: {X_basic_plus_clin_sdh.shape[1]}")
print(f"   - Total cells: {X_basic_plus_clin_sdh.shape[0] * X_basic_plus_clin_sdh.shape[1]}")

print("\n2. DATA TYPES:")
print(X_basic_plus_clin_sdh.dtypes.value_counts())

print("\n3. MISSING VALUES:")
missing_counts = X_basic_plus_clin_sdh.isna().sum()
if missing_counts.sum() == 0:
    print("   - No missing values")
else:
    print(missing_counts[missing_counts > 0])

print("\n4. DESCRIPTIVE STATISTICS:")
print(X_basic_plus_clin_sdh.describe().T)

print("\n5. CORRELATIONS WITH RESPONSE VARIABLES:")
print(f"   - Correlation with log_wer:")
print(X_basic_plus_clin_sdh.corr()['log_wer'].sort_values(ascending=False).head(10))
print(f"\n   - Correlation with sentCoherenceSentBertCumulativeCentroid:")
print(X_basic_plus_clin_sdh.corr()['sentCoherenceSentBertCumulativeCentroid'].sort_values(ascending=False).head(10))

print("\n6. MEMORY USAGE:")
print(f"   - Total memory: {X_basic_plus_clin_sdh.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "=" * 80)

SUMMARY STATISTICS: X_basic_plus_clin_sdh

1. DATASET SHAPE AND SIZE:
   - Rows: 2863
   - Columns: 32
   - Total cells: 91616

2. DATA TYPES:
float64    31
int64       1
Name: count, dtype: int64

3. MISSING VALUES:
   - No missing values

4. DESCRIPTIVE STATISTICS:
                                          count         mean         std  \
pid                                      2863.0  1313.323088  696.133772   
race_2.0                                 2863.0     0.166958    0.373003   
race_4.0                                 2863.0     0.005239    0.072205   
race_5.0                                 2863.0     0.036675    0.187995   
race_6.0                                 2863.0     0.138666    0.345658   
race_999.0                               2863.0     0.004541    0.067243   
gender_2.0                               2863.0     0.453720    0.497941   
gender_3.0                               2863.0     0.051345    0.220738   
gender_4.0                               2863.0 

In [33]:
# --- Merge Geographic Location Data into X_basic_plus_clin_sdh ---
import pandas as pd
import os

# Load the geographic location data
location_path = '/home/NETID/obdw4/location_data_patrick/olivermergedcounts.csv'
try:
    location_df = pd.read_csv(location_path)
    print(f"Loaded location data: {location_df.shape}")
    print(f"Location data columns: {list(location_df.columns)}")
    
    # Filter to primary cluster only
    if 'is_primary_cluster' in location_df.columns:
        location_df_primary = location_df[location_df['is_primary_cluster'] == True].copy()
        print(f"Filtered to primary clusters only: {location_df_primary.shape}")
        print(f"Unique participants in location data: {location_df_primary['account_id'].nunique()}")
    else:
        print("WARNING: 'is_primary_cluster' column not found, using all location data")
        location_df_primary = location_df.copy()
    
    # Select only essential geographic columns
    # Keep: account_id (for merge), SVI indices (RPL_THEMES + subscores), RUCA codes, and STATEFP
    essential_columns = [
        'account_id',           # For merging with pid
        'RPL_THEMES',           # Overall SVI percentile
        'RPL_THEME1',           # Socioeconomic SVI subscore
        'RPL_THEME2',           # Household composition SVI subscore
        'RPL_THEME3',           # Minority status/language SVI subscore
        'RPL_THEME4',           # Housing/transportation SVI subscore
        'PrimaryRUCA',          # Urban/rural classification (primary)
        'SecondaryRUCA',        # Urban/rural classification (secondary)
        'STATEFP'               # State FIPS code
    ]
    
    # Filter to only columns that exist in the dataframe
    available_columns = [col for col in essential_columns if col in location_df_primary.columns]
    missing_columns = [col for col in essential_columns if col not in location_df_primary.columns]
    
    if missing_columns:
        print(f"\nWARNING: Missing expected columns: {missing_columns}")
    
    print(f"\nSelecting {len(available_columns)} essential geographic columns:")
    print(f"  {available_columns}")
    
    # Keep only essential columns
    location_df_primary = location_df_primary[available_columns]
    print(f"Reduced location data shape: {location_df_primary.shape}")
    
    # Dropped columns summary
    print(f"\nDropped columns:")
    print(f"  - Google Maps count variables (doctor_count, pharmacy_count, etc.)")
    print(f"  - Demographic/clinical duplicates (age, gender, race, clinical scores, etc.)")
    print(f"  - Cluster statistics (cluster_rank, total_minutes, etc.)")
    print(f"  - Raw coordinates (medoid_lat/lon, latitude/longitude)")
    print(f"  - Event tracking data (event_data, event_time)")
    print(f"  - Geographic identifiers (COUNTYFP, TRACTFP, GEOID)")
        
except Exception as e:
    print(f"Could not load location data: {e}")
    location_df_primary = None

# Display columns in X_basic_plus_clin_sdh
print(f"\nX_basic_plus_clin_sdh shape: {X_basic_plus_clin_sdh.shape}")
print(f"Unique participants in X_basic_plus_clin_sdh: {X_basic_plus_clin_sdh['pid'].nunique()}")

# Merge on account_id = pid
if location_df_primary is not None:
    if 'account_id' not in location_df_primary.columns:
        print("ERROR: 'account_id' not found in location data.")
    elif 'pid' not in X_basic_plus_clin_sdh.columns:
        print("ERROR: 'pid' not found in X_basic_plus_clin_sdh.")
    else:
        print(f"\nMerging on: location_df['account_id'] = X_basic_plus_clin_sdh['pid']")
        
        # Merge on PID only (one primary cluster per participant)
        merged_df = X_basic_plus_clin_sdh.merge(
            location_df_primary, 
            left_on='pid', 
            right_on='account_id', 
            how='left'
        )
        
        # Remove duplicate rows from merged_df
        merged_df = merged_df.drop_duplicates()
        
        print(f"\nMerged DataFrame shape: {merged_df.shape}")
        print(f"Original X_basic_plus_clin_sdh shape: {X_basic_plus_clin_sdh.shape}")
        print(f"Rows with location data: {merged_df['account_id'].notna().sum()}")
        print(f"Rows without location data: {merged_df['account_id'].isna().sum()}")
        
        # Save
        merged_df.to_csv(out_folder + "X_basic_plus_clin_sdh_with_location.csv")
        print("\nSaved merged DataFrame with location data.")
else:
    print("Location data not loaded; skipping merge.")

Loaded location data: (1038, 114)
Location data columns: ['medoid_lat', 'medoid_lon', 'cluster', 'account_id', 'doctor_count', 'pharmacy_count', 'gym_count', 'library_count', 'park_count', 'bus_station_count', 'transit_station_count', 'supermarket_count', 'event_data', 'event_time', 'longitude', 'latitude', 'STATEFP', 'COUNTYFP', 'TRACTFP', 'TRACT_GEOID', 'GEOID', 'PrimaryRUCA', 'SecondaryRUCA', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4', 'RPL_THEMES', 'in_person', 'account_status', 'page_progress', 'days of data', 'Tracking', 'age', 'gender', 'phone', 'data_plan', 'ah_freq', 'location', 'availability', 'contact', 'comp_q1', 'comp_q2', 'comp_q3', 'comp_q4', 'comp_q5', 'race', 'hispanic', 'marital-status', 'sexuality', 'education', 'employment-status', 'living-status', 'basic-cell', 'smartphone', 'tablet', 'computer', 'wearable-tech', 'email', 'social-media', 'verify-hearing-voices', 'how-often', 'how-long', 'diagnoses', 'used-treatment', 'type-of-treatments', 'who-knows', '

### Clean and Encode Location Data
Remove duplicate columns from the location merge and one-hot encode categorical geographic variables

In [34]:
# Clean merged dataframe with location data
if 'merged_df' in locals():
    print("Original merged_df shape:", merged_df.shape)
    
    # Identify duplicate columns (columns that existed in X_basic_plus_clin_sdh)
    original_cols = set(X_basic_plus_clin_sdh.columns)
    location_cols = set(merged_df.columns) - original_cols
    duplicate_cols = location_cols.intersection(original_cols)
    
    print(f"\nDuplicate columns to drop: {len(duplicate_cols)}")
    if len(duplicate_cols) > 0:
        print(f"Examples: {list(duplicate_cols)[:10]}")
    
    # Also drop account_id (duplicate of pid)
    cols_to_drop = list(duplicate_cols) + ['account_id']
    cols_to_drop = [col for col in cols_to_drop if col in merged_df.columns]
    
    # Drop duplicate columns
    X_location_clean = merged_df.drop(columns=cols_to_drop, errors='ignore')
    print(f"\nAfter dropping duplicates: {X_location_clean.shape}")
    
    # Remove columns with 0 or 1 unique values (causes R regression errors)
    single_value_cols = [col for col in X_location_clean.columns if X_location_clean[col].nunique() <= 1]
    if len(single_value_cols) > 0:
        print(f"\nRemoving {len(single_value_cols)} columns with 0 or 1 unique values:")
        print(f"  {single_value_cols}")
        X_location_clean = X_location_clean.drop(columns=single_value_cols)
        print(f"After removing constant columns: {X_location_clean.shape}")
    
    # Identify categorical location columns to encode
    categorical_location_cols = []
    
    # Check which categorical columns exist
    potential_categorical = ['PrimaryRUCA', 'SecondaryRUCA', 'STATEFP']
    for col in potential_categorical:
        if col in X_location_clean.columns:
            categorical_location_cols.append(col)
            print(f"\n{col} unique values: {X_location_clean[col].nunique()}")
    
    # One-hot encode categorical location variables
    if len(categorical_location_cols) > 0:
        print(f"\nOne-hot encoding: {categorical_location_cols}")
        X_location_encoded = one_hot_encode(df=X_location_clean, cols2encode=categorical_location_cols)
        
        # Drop original categorical columns and add encoded versions
        X_location_final = X_location_clean.drop(columns=categorical_location_cols)
        X_location_final = pd.concat([X_location_final, X_location_encoded], axis=1)
        
        print(f"\nFinal shape after encoding: {X_location_final.shape}")
    else:
        print("\nNo categorical location columns found to encode")
        X_location_final = X_location_clean
    
    # Save final dataset
    X_location_final.to_csv(out_folder + "X_basic_plus_clin_sdh_location_encoded.csv")
    print("\nSaved X_basic_plus_clin_sdh_location_encoded.csv")
    
    # Display column summary
    print(f"\nFinal dataset columns: {X_location_final.shape[1]}")
    location_specific_cols = [col for col in X_location_final.columns if col not in X_basic_plus_clin_sdh.columns]
    print(f"Location-specific columns added: {len(location_specific_cols)}")
    print(f"Examples: {location_specific_cols[:20]}")
else:
    print("No merged_df found. Please run the previous cell first.")

Original merged_df shape: (2863, 41)

Duplicate columns to drop: 0

After dropping duplicates: (2863, 40)

PrimaryRUCA unique values: 10

SecondaryRUCA unique values: 12

STATEFP unique values: 39

One-hot encoding: ['PrimaryRUCA', 'SecondaryRUCA', 'STATEFP']
[array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., nan]), array([ 1. ,  1.1,  2. ,  3. ,  4. ,  4.1,  5. ,  6. ,  7.1,  8. ,  9. ,
       10. ,  nan]), array([ 1.,  4.,  5.,  6.,  8.,  9., 12., 13., 16., 17., 18., 19., 20.,
       21., 22., 24., 25., 26., 27., 28., 29., 32., 33., 34., 35., 36.,
       37., 39., 40., 41., 42., 44., 47., 48., 49., 50., 51., 53., 55.,
       nan])]
['PrimaryRUCA_2.0' 'PrimaryRUCA_3.0' 'PrimaryRUCA_4.0' 'PrimaryRUCA_5.0'
 'PrimaryRUCA_6.0' 'PrimaryRUCA_7.0' 'PrimaryRUCA_8.0' 'PrimaryRUCA_9.0'
 'PrimaryRUCA_10.0' 'PrimaryRUCA_nan' 'SecondaryRUCA_1.1'
 'SecondaryRUCA_2.0' 'SecondaryRUCA_3.0' 'SecondaryRUCA_4.0'
 'SecondaryRUCA_4.1' 'SecondaryRUCA_5.0' 'SecondaryRUCA_6.0'
 'SecondaryRUCA_7.1' 'Sec